# 1. User Clarification and Brief Generation

This notebook demonstrates how to scope research requests using structured outputs and date-aware prompts with LangGraph.

In [ ]:
import os
from datetime import datetime
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

## 1. LLM Initialization and Pydantic Schemas

In [ ]:
def get_llm(temperature: float = 0.2):
    return ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
        temperature=temperature,
    )

class ClarifyWithUser(BaseModel):
    needs_clarification: bool = Field(
        description="True if the request is ambiguous or underspecified."
    )
    clarification_questions: List[str] = Field(
        default_factory=list,
        description="1 to 3 targeted clarification questions.",
    )
    reasoning: str = Field(description="Reasoning behind the decision.")

class ResearchQuestion(BaseModel):
    title: str = Field(description="Descriptive title of the research project.")
    core_question: str = Field(description="Core question to be investigated.")
    sub_topics: List[str] = Field(description="3 to 5 sub-topics for parallel investigation.")
    target_depth: str = Field(default="in-depth", description="Depth: overview, in-depth, or comprehensive.")
    search_queries: List[str] = Field(description="Initial search queries.")
    key_requirements: List[str] = Field(default_factory=list, description="Specific requirements.")

## 2. Graph State & Node Definitions

In [ ]:
class ScopingState(BaseModel):
    user_query: str
    user_response: Optional[str] = None
    clarification: Optional[ClarifyWithUser] = None
    research_brief: Optional[ResearchQuestion] = None
    status: str = "initialized"

async def check_clarification_node(state: ScopingState) -> Dict[str, Any]:
    today = datetime.now().strftime("%B %d, %Y")
    system_prompt = f"You are a Research Director. Today is {today}. Analyze if clarification is needed."
    
    llm = get_llm(temperature=0.1).with_structured_output(ClarifyWithUser)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"User Request: {state.user_query}"),
    ]
    clarification = await llm.ainvoke(messages)
    return {
        "clarification": clarification,
        "status": "clarification_needed" if clarification.needs_clarification else "ready_for_brief"
    }

async def generate_brief_node(state: ScopingState) -> Dict[str, Any]:
    today = datetime.now().strftime("%B %d, %Y")
    system_prompt = f"You are a Research Architect. Today is {today}. Generate a structured research brief."
    
    context = f"User Query: {state.user_query}"
    if state.user_response:
        context += f"\nClarification Answers: {state.user_response}"
        
    llm = get_llm(temperature=0.2).with_structured_output(ResearchQuestion)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=context),
    ]
    brief = await llm.ainvoke(messages)
    return {"research_brief": brief, "status": "completed"}

def route_scoping(state: ScopingState) -> str:
    if state.clarification and state.clarification.needs_clarification and not state.user_response:
        return "wait_for_user"
    return "generate_brief"

## 3. Build and Test Scoping Graph

In [ ]:
builder = StateGraph(ScopingState)
builder.add_node("check_clarification", check_clarification_node)
builder.add_node("generate_brief", generate_brief_node)

builder.add_edge(START, "check_clarification")
builder.add_conditional_edges(
    "check_clarification",
    route_scoping,
    {
        "wait_for_user": END,
        "generate_brief": "generate_brief",
    }
)
builder.add_edge("generate_brief", END)
scoping_graph = builder.compile()

# Test Scoping Graph
state_input = ScopingState(user_query="Analyze the current progress in quantum computing hardware architectures.")
result = await scoping_graph.ainvoke(state_input)
print("Status:", result["status"])
if result.get("research_brief"):
    print("Brief Title:", result["research_brief"].title)
    print("Sub-Topics:", result["research_brief"].sub_topics)